# Create validation rules via the API — `POST /validation-rules`

Adds a validation rule through the API (no editing of `seed_validation_rules.py`,
no server restart) and checks it fires. The worked example uses the **`Romanize`**
data type: *the location `Name` must be Latin/ASCII only.*

**Before running:**
1. `python run.py` — API on http://localhost:8000
2. Run `reset_db.ipynb` for a clean slate (entities skip on re-POST and aren't
   re-validated). The rule tables can keep their contents — `POST /validation-rules`
   is `INSERT OR REPLACE` per `rule_id` / `check_id` / `logic_id`.

The endpoint is deliberately permissive about shape — section 3 posts the *same*
rule four different ways and shows they all land identically.

In [2]:
import json
import requests
import pandas as pd

BASE_URL = "http://localhost:8000"
RULES_URL = f"{BASE_URL}/validation-rules"


def create_rule(body):
    """POST any accepted shape to /validation-rules; return the summary."""
    r = requests.post(RULES_URL, json=body)
    if r.status_code >= 400:
        raise RuntimeError(f"{r.status_code}: {r.text}")
    return r.json()


def get_rule_defs():
    return requests.get(RULES_URL).json()


# --- sanity check: is this server running the code with /validation-rules? ---
spec = requests.get(f"{BASE_URL}/openapi.json").json()
if "/validation-rules" not in spec.get("paths", {}):
    raise RuntimeError(
        "The server on :8000 has no /validation-rules route — it is running code "
        "from before this endpoint was added.\n"
        "  1. sync entity_api/rule_authoring.py, entity_api/routers/validation_rules.py "
        "and entity_api/main.py to this machine\n"
        "  2. fully STOP and restart `python run.py` (a new router module is not always "
        "picked up by --reload)\n"
        "  3. if it fails to start with \"cannot import name 'AliasChoices'\", that env "
        "has Pydantic v1 -> pip install -U \"pydantic>=2\" \"fastapi>=0.110\""
    )
print("OK - /validation-rules is registered. routes:",
      sorted(p for p in spec["paths"] if "validation" in p or p == "/entities"))


OK - /validation-rules is registered. routes: ['/entities', '/entities/validation', '/entities/validation-results', '/validation-rules']


## 1. Create the Romanize rule

Posted with camelCase keys and a **nested condition** (`field` / `dataType` are
accepted aliases for `field_name` / `data_type`). The engine reads it on the next
request — no restart.

In [3]:
romanize_rule = {
    "ruleId": "API-ROMANIZE-NAME",
    "validationType": "Location Master Data - Romanization",
    "description": "Location name must be Latin/ASCII only",
    "severity": "Warning",
    "onExceptionDescription": "Location name is not romanized",
    "conditions": [
        {"role": "Target", "field": "Name", "dataType": "Romanize"}
    ],
}

print(json.dumps(create_rule(romanize_rule), indent=2))

{
  "rules_written": [
    "API-ROMANIZE-NAME"
  ],
  "conditions_written": 1,
  "checks_written": [],
  "logic_written": [],
  "generated_ids": [],
  "warnings": []
}


In [ ]:
# confirm it landed in val_rules + val_rule_conditions
defs = get_rule_defs()
display(pd.DataFrame(defs["rules"]))
display(pd.DataFrame([c for c in defs["conditions"] if c["rule_id"] == "API-ROMANIZE-NAME"]))

,rule_id,validation_type,function_name,check_id,description,severity,active,on_exception_status,on_exception_description
0,API-ROMANIZE-NAME,Location Master Data - Romanization,None,None,Location name must be Latin/ASCII only,Warning,Y,-1,Location name is not romanized


,id,rule_id,role,field_name,operator,expected_value,data_type,constraint_expr,reference_source,logical_group
0,1,API-ROMANIZE-NAME,Target,Name,None,None,Romanize,None,None,None


: 

## 2. Feed data and watch the rule fire

`Name` comes from `entities_location`, joined to each entity on
`(systemCode, businessEntityCode)`. Two of the three location names are non-Latin.

In [ ]:
locations = [
    {"systemCode": "RMZ", "Code": "ROM-CYR", "Country": "BG",
     "City": "Shumen", "Name": "Шумен Warehouse"},   # Cyrillic
    {"systemCode": "RMZ", "Code": "ROM-CJK", "Country": "SG",
     "City": "Singapore", "Name": "上海 Depot"},                        # Chinese
    {"systemCode": "RMZ", "Code": "ROM-LAT", "Country": "MY",
     "City": "Shah Alam", "Name": "Shah Alam Plant"},                          # Latin
]
print("POST locations:", requests.post(f"{BASE_URL}/entities/inputlocation", json=locations).status_code)

entities = {"mappingsInformation": [
    {"systemCode": "RMZ", "businessEntityCode": "ROM-CYR", "EOID": "E-CYR", "FID": "F-CYR"},
    {"systemCode": "RMZ", "businessEntityCode": "ROM-CJK", "EOID": "E-CJK", "FID": "F-CJK"},
    {"systemCode": "RMZ", "businessEntityCode": "ROM-LAT", "EOID": "E-LAT", "FID": "F-LAT"},
]}
proc = requests.post(f"{BASE_URL}/entities/process", json=entities).json()
print("POST entities :", json.dumps(proc, indent=2))

if proc.get("inserted", 0) == 0:
    print("\n!! nothing was inserted (entities already exist). The results below are\n"
          "   from a previous run — run reset_db.ipynb, then re-run this notebook.")

In [ ]:
vr = pd.DataFrame(requests.get(f"{BASE_URL}/entities/validation-results").json())
display(vr[vr["rule_id"] == "API-ROMANIZE-NAME"][
    ["businessEntityCode", "rule_id", "passed", "severity", "status", "description"]
])

Expected: `ROM-CYR` and `ROM-CJK` **fail** (`passed = 0`, "Location name is not
romanized"); `ROM-LAT` **passes**.

## 3. Format variations — same rule, four ways

Each payload below is parsed and mapped to the same `val_rules` + `val_checks` /
`val_rule_conditions` shape.

In [ ]:
variants = {
    "flat shortcut (no 'conditions' block)": {
        "rule_id": "API-ROMANIZE-V2",
        "description": "name must romanize",
        "field": "Name",
        "data_type": "Romanize",
    },
    "inline check object": {
        "rule_id": "API-ROMANIZE-V3",
        "check": {"type": "Data Type", "fields": ["Name"], "value": "Romanize"},
    },
    "seed layout + positional arrays": {
        "RULES": [
            ["API-ROMANIZE-V4", "Romanization", None, "chk_rom_v4",
             "name must romanize", "Warning", "Y", -1, "not romanized"]
        ],
        "CHECKS": [
            ["chk_rom_v4", "Data Type", "Name", "Romanize", None, None, "AND"]
        ],
    },
    "bare list, camelCase": [
        {"ruleId": "API-ROMANIZE-V5", "fieldName": "Name", "dataType": "Romanize"}
    ],
}

for label, body in variants.items():
    print(f"{label:<40} -> {create_rule(body)}")

In [ ]:
# all five API-ROMANIZE-* rules, side by side
defs = get_rule_defs()
rom_rules = [r for r in defs["rules"] if r["rule_id"].startswith("API-ROMANIZE")]
display(pd.DataFrame(rom_rules))

rom_ids = {r["rule_id"] for r in rom_rules}
print("conditions:")
display(pd.DataFrame([c for c in defs["conditions"] if c["rule_id"] in rom_ids]))
print("checks:")
display(pd.DataFrame([c for c in defs["checks"] if c["check_id"] in (
    {r["check_id"] for r in rom_rules if r["check_id"]})]))

## Notes

- **Idempotent:** re-POSTing the same `rule_id` overwrites (`INSERT OR REPLACE`).
  Conditions have no natural key, so *all* conditions for a `rule_id` present in the
  payload are replaced.
- **No restart:** `loader` re-reads the rule tables on every `/entities/process`.
- **Auto-IDs:** omit `check_id` / `logic_id` / `rule_id` and one is generated
  (`api-check-…`), returned in `generated_ids`.
- A rule with neither a `check_id` nor any condition is accepted but always passes —
  the response lists it under `warnings`.
- Field names in rules are case-insensitive (`Name` == `name`).
- Re-running this notebook: run `reset_db.ipynb` first, otherwise the entities are
  `skipped / already_exists` and nothing is re-validated.